# 13. Experiment 1 — Corpus statistics

In [12]:
import gzip
import json
import nltk
import matplotlib.pyplot as plt
from collections import Counter
from nltk.tokenize import sent_tokenize, word_tokenize
from nltk.util import ngrams
from tqdm import tqdm

# Tải data của NLTK (chỉ cần chạy lần đầu)
nltk.download('punkt')
nltk.download('punkt_tab')

def process_c4_corpus(filepath, max_docs=None):
    num_docs = 0
    num_sentences = 0
    num_tokens = 0
    
    unigram_counter = Counter()
    bigram_counter = Counter()
    trigram_counter = Counter()
    
    print(f"Đang đọc và xử lý dữ liệu từ: {filepath}")
    
    # Mở file .json.gz
    with gzip.open(filepath, 'rt', encoding='utf-8') as f:
        for line in tqdm(f, total=30000, desc="Processing Documents"):
            if max_docs and num_docs >= max_docs:
                break
                
            # Đọc JSON object từ dòng hiện tại
            data = json.loads(line)
            text = data.get("text", "")
            if not text:
                continue
                
            num_docs += 1
            
            # Tách câu
            sentences = sent_tokenize(text)
            num_sentences += len(sentences)
            
            # Tách từ và tạo n-grams
            for sent in sentences:
                tokens = [word.lower() for word in word_tokenize(sent) if word.isalnum()]
                num_tokens += len(tokens)
                
                unigram_counter.update(tokens)
                bigram_counter.update(ngrams(tokens, 2))
                trigram_counter.update(ngrams(tokens, 3))

    # Đếm số lượng n-gram chỉ xuất hiện 1 lần
    once_unigrams = sum(1 for count in unigram_counter.values() if count == 1)
    once_bigrams = sum(1 for count in bigram_counter.values() if count == 1)
    once_trigrams = sum(1 for count in trigram_counter.values() if count == 1)
    
    stats = {
        "num_docs": num_docs,
        "num_sentences": num_sentences,
        "num_tokens": num_tokens,
        "vocab_size": len(unigram_counter),
        "unique_unigrams": len(unigram_counter),
        "unique_bigrams": len(bigram_counter),
        "unique_trigrams": len(trigram_counter),
        "once_unigrams": once_unigrams,
        "once_bigrams": once_bigrams,
        "once_trigrams": once_trigrams,
    }
    
    return stats, unigram_counter, bigram_counter, trigram_counter

def plot_freq_distribution(counter, title, filename):
    """Vẽ biểu đồ phân bố tần suất trên thang log-log"""
    counts = sorted(counter.values(), reverse=True)
    
    plt.figure(figsize=(10, 6))
    plt.plot(counts, color='b')
    plt.xscale('log')
    plt.yscale('log')
    plt.title(title)
    plt.xlabel('Rank (log scale)')
    plt.ylabel('Frequency (log scale)')
    plt.grid(True, which="both", ls="--", alpha=0.5)
    plt.tight_layout()
    plt.savefig(filename)
    plt.close()
    print(f"Đã lưu biểu đồ: {filename}")

# --- KHỞI CHẠY ---
file_path = r"C:\Users\tasdus_117\Desktop\NLP\lab02\c4-train.00000-of-01024-30K.json.gz"

# Chạy thống kê và xử lý n-grams
stats, unigrams, bigrams, trigrams = process_c4_corpus(file_path)

# In kết quả dạng bảng
print("\n--- CORPUS STATISTICS ---")
print(f"Số documents: {stats['num_docs']:,}")
print(f"Số sentences: {stats['num_sentences']:,}")
print(f"Số tokens:    {stats['num_tokens']:,}")
print(f"Vocabulary:   {stats['vocab_size']:,}\n")

print(f"{'Model':<10} | {'# unique n-grams':<18} | {'# n-grams xuất hiện 1 lần':<25}")
print("-" * 59)
print(f"{'Unigram':<10} | {stats['unique_unigrams']:<18,} | {stats['once_unigrams']:<25,}")
print(f"{'Bigram':<10} | {stats['unique_bigrams']:<18,} | {stats['once_bigrams']:<25,}")
print(f"{'Trigram':<10} | {stats['unique_trigrams']:<18,} | {stats['once_trigrams']:<25,}")

# Vẽ và lưu biểu đồ vào cùng thư mục chạy script
plot_freq_distribution(unigrams, 'Unigram Frequency Distribution (C4 Dataset)', 'unigram_freq.png')
plot_freq_distribution(bigrams, 'Bigram Frequency Distribution (C4 Dataset)', 'bigram_freq.png')
plot_freq_distribution(trigrams, 'Trigram Frequency Distribution (C4 Dataset)', 'trigram_freq.png')

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\tasdus_117\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\tasdus_117\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


Đang đọc và xử lý dữ liệu từ: C:\Users\tasdus_117\Desktop\NLP\lab02\c4-train.00000-of-01024-30K.json.gz


Processing Documents: 100%|██████████| 30000/30000 [00:36<00:00, 825.57it/s]



--- CORPUS STATISTICS ---
Số documents: 30,000
Số sentences: 622,194
Số tokens:    10,692,174
Vocabulary:   180,774

Model      | # unique n-grams   | # n-grams xuất hiện 1 lần
-----------------------------------------------------------
Unigram    | 180,774            | 86,013                   
Bigram     | 2,640,066          | 1,886,578                
Trigram    | 6,225,752          | 5,371,879                
Đã lưu biểu đồ: unigram_freq.png
Đã lưu biểu đồ: bigram_freq.png
Đã lưu biểu đồ: trigram_freq.png
